# Week 4

This is a two-tab Gradio app that uses LLMs to work with code.
- Tab 1: Python to C++ Converter: Paste Python, pick a provider/model, convert it to optimised C++, then run both side-by-side to compare speed.
- Tab 2: Docstring Adder: Paste any code, select the language, and the model adds module, function, and class docstrings plus inline comments — without touching the logic.

In [ ]:
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from huggingface_hub import InferenceClient

In [ ]:
# ── Env & clients ────────────────────────────────────────────────────────────
load_dotenv(override=True)

openai_api_key     = os.getenv("OPENAI_API_KEY")
ollama_api_key     = os.getenv("OLLAMA_API_KEY")
anthropic_api_key  = os.getenv("ANTHROPIC_API_KEY")
google_api_key     = os.getenv("GOOGLE_API_KEY")
grok_api_key       = os.getenv("GROK_API_KEY")
groq_api_key       = os.getenv("GROQ_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
hugging_face_token = os.getenv("HUGGING_FACE_TOKEN")

def _clean_key(k):
    if not k:
        return None
    return k.strip().strip('"').strip("'")

hugging_face_token = _clean_key(hugging_face_token)

for name, key in [
    ("OpenAI",       openai_api_key),
    ("Anthropic",    anthropic_api_key),
    ("Google",       google_api_key),
    ("Grok",         grok_api_key),
    ("Groq",         groq_api_key),
    ("OpenRouter",   openrouter_api_key),
    ("HuggingFace",  hugging_face_token),
]:
    if key:
        print(f"{name} key exists and begins with '{key[:6]}'")
    else:
        print(f"{name} key not set (optional)")

In [ ]:
# ── OpenAI-compatible clients ─────────────────────────────────────────────────
openai_client    = OpenAI()
anthropic        = OpenAI(api_key=anthropic_api_key,  base_url="https://api.anthropic.com/v1/")
gemini           = OpenAI(api_key=google_api_key,     base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
grok             = OpenAI(api_key=grok_api_key,       base_url="https://api.x.ai/v1")
groq_client      = OpenAI(api_key=groq_api_key,       base_url="https://api.groq.com/openai/v1")
ollama           = OpenAI(api_key=ollama_api_key,     base_url="http://localhost:11434/v1")
openrouter       = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")
hf               = InferenceClient(token=hugging_face_token) if hugging_face_token else None

In [ ]:
# ── Model registry ────────────────────────────────────────────────────────────
MODEL_OPTIONS = {
    "OpenAI":       ["gpt-5", "gpt-5.4-2026-03-05", "gpt-4.1-2025-04-14"],
    "Anthropic":    ["claude-sonnet-4-5-20250929", "claude-haiku-4-5"],
    "Gemini":       ["gemini-2.5-pro", "gemini-2.5-flash-lite"],
    "Grok":         ["grok-4", "grok-4-fast-non-reasoning"],
    "Groq":         ["openai/gpt-oss-120b"],
    "Ollama":       ["llama3.2", "qwen3.8b", "qwen3-coder:30b", "deepseek-r1:8b"],
    "OpenRouter":   ["qwen/qwen3-coder-plus"],
    "HuggingFace":  ["Qwen/Qwen2.5-14B-Instruct"],
}

CLIENTS = {
    "OpenAI":       openai_client,
    "Anthropic":    anthropic,
    "Gemini":       gemini,
    "Grok":         grok,
    "Groq":         groq_client,
    "Ollama":       ollama,
    "OpenRouter":   openrouter,
}


compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native",
                   "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────────
def update_model_dropdown(provider: str):
    choices = MODEL_OPTIONS.get(provider, [])
    return gr.Dropdown(choices=choices, value=choices[0] if choices else None)


def _call_model(provider: str, model: str, messages: list) -> str:
    """Route a chat-completion request to the right client/provider."""
    if provider == "HuggingFace":
        if hf is None:
            return "Error: HuggingFace token not set."
        response = hf.chat_completion(
            model=model,
            messages=messages,
            max_tokens=4096,
        )
        return response.choices[0].message.content
    
    client = CLIENTS[provider]
    reasoning_effort = "high" if "gpt" in model.lower() else None
    kwargs = dict(model=model, messages=messages)
    if reasoning_effort:
        kwargs["reasoning_effort"] = reasoning_effort
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content

In [ ]:
# ── Tab 1: Python → C++ code converter
system_prompt_convert = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def _user_prompt_convert(python: str) -> str:
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output.
Your response will be written to main.cpp and compiled with:
{compile_command}
Respond only with C++ code.

```python
{python}
```
"""

def port_to_cpp(provider: str, model: str, python: str) -> str:
    messages = [
        {"role": "system",  "content": system_prompt_convert},
        {"role": "user",    "content": _user_prompt_convert(python)},
    ]
    reply = _call_model(provider, model, messages)
    return reply.replace("```cpp", "").replace("```", "")


def run_python(code: str) -> str:
    buf = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buf
    try:
        exec(code, {"__builtins__": __builtins__})
        return buf.getvalue()
    except Exception as e:
        return f"Error: {e}"
    finally:
        sys.stdout = old_stdout


def compile_and_run(cpp_code: str) -> str:
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp_code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return result.stdout
    except subprocess.CalledProcessError as e:
        return f"Compilation/run error:\n{e.stderr}"

In [ ]:
# ── Tab 2: Docstring / comment adder ─────────────────────────────────────────
SUPPORTED_LANGUAGES = ["Python", "C++", "Rust", "JavaScript", "TypeScript", "Java"]

DOCSTRING_STYLE = {
    "Python":     "Google-style docstrings (Args:, Returns:, Raises:, Example:)",
    "C++":        "Doxygen-style comments (/// @brief, @param, @return)",
    "Rust":       "Rust doc comments (///, # Examples, # Panics, # Errors)",
    "JavaScript": "JSDoc comments (@param, @returns, @throws, @example)",
    "TypeScript": "JSDoc/TSDoc comments (@param, @returns, @throws, @example)",
    "Java":       "Javadoc comments (@param, @return, @throws, @since)",
}

system_prompt_docstring = """
You are an expert code documentation assistant.
Your task is to add clear, professional docstrings and inline comments to the provided code.
- Add module/file-level docstrings where appropriate.
- Add function/method docstrings using the style specified.
- Add class-level docstrings.
- Add concise inline comments for non-obvious logic.
- Do NOT change the code logic — only add documentation.
- Respond with the fully documented code only. No explanations outside the code.
"""

def _user_prompt_docstring(code: str, language: str) -> str:
    style = DOCSTRING_STYLE.get(language, "standard documentation comments")
    return f"""
Add comprehensive docstrings and comments to the following {language} code.
Use {style}.
Add:
  1. A module/file-level docstring at the top summarising what the code does.
  2. A docstring for every function, method, and class.
  3. Inline comments for any non-obvious logic.

Return the fully documented {language} code and nothing else.

```{language.lower()}
{code}
```
"""

def add_docstrings(provider: str, model: str, code: str, language: str) -> str:
    if not code.strip():
        return "Please paste some code first."
    messages = [
        {"role": "system", "content": system_prompt_docstring},
        {"role": "user",   "content": _user_prompt_docstring(code, language)},
    ]
    reply = _call_model(provider, model, messages)
    # Strip any markdown fences the model might have added
    for fence in [f"```{language.lower()}", "```python", "```cpp", "```rust",
                  "```javascript", "```typescript", "```java", "```"]:
        reply = reply.replace(fence, "")
    return reply.strip()

In [ ]:
# ── Sample code snippets ──────────────────────────────────────────────────────
PI_PYTHON = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
""".strip()

SAMPLE_UNDOCUMENTED = """
def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value

def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum
""".strip()

In [ ]:
# ── Gradio UI ─────────────────────────────────────────────────────────────────
with gr.Blocks(title="Code Tools — Week 4") as ui:
    gr.Markdown("# Code Tools — Week 4\nTwo tools: **Python to C++ converter** and **Docstring adder**.")

    # ── Shared model selector (reused across tabs) ────────────────────────────
    # Each tab has its own provider/model row for independence.

    # ── Tab 1: Python to C++ ──────────────────────────────────────────────────
    with gr.Tab("Python to C++"):
        gr.Markdown("Convert Python to optimised C++ and optionally run both.")

        with gr.Row():
            cpp_provider = gr.Dropdown(
                choices=list(MODEL_OPTIONS.keys()),
                value="OpenAI",
                label="Provider"
            )
            cpp_model = gr.Dropdown(
                choices=MODEL_OPTIONS["OpenAI"],
                value=MODEL_OPTIONS["OpenAI"][0],
                label="Model"
            )

        cpp_provider.change(fn=update_model_dropdown, inputs=[cpp_provider], outputs=[cpp_model])

        with gr.Row():
            python_code = gr.Code(label="Python (original)", value=PI_PYTHON,
                                  language="python", lines=22)
            cpp_code    = gr.Code(label="C++ (generated)",   value="",
                                  language="cpp",    lines=22)

        with gr.Row():
            py_run_btn  = gr.Button("▶ Run Python")
            convert_btn = gr.Button("⚡ Convert to C++")
            cpp_run_btn = gr.Button("▶ Compile & Run C++")

        with gr.Row():
            py_out  = gr.TextArea(label="Python output",  lines=6)
            cpp_out = gr.TextArea(label="C++ output",     lines=6)

        convert_btn.click(fn=port_to_cpp,       inputs=[cpp_provider, cpp_model, python_code], outputs=[cpp_code])
        py_run_btn.click( fn=run_python,         inputs=[python_code],                          outputs=[py_out])
        cpp_run_btn.click(fn=compile_and_run,    inputs=[cpp_code],                             outputs=[cpp_out])


    # ── Tab 2: Docstring Adder ────────────────────────────────────────────────
    with gr.Tab("Docstring Adder"):
        gr.Markdown(
            "Paste any code below and choose a language — the model will add "
            "module, function, class docstrings **and** inline comments without "
            "changing the logic."
        )

        with gr.Row():
            doc_provider = gr.Dropdown(
                choices=list(MODEL_OPTIONS.keys()),
                value="OpenAI",
                label="Provider"
            )
            doc_model = gr.Dropdown(
                choices=MODEL_OPTIONS["OpenAI"],
                value=MODEL_OPTIONS["OpenAI"][0],
                label="Model"
            )
            doc_language = gr.Dropdown(
                choices=SUPPORTED_LANGUAGES,
                value="Python",
                label="Language"
            )

        doc_provider.change(fn=update_model_dropdown, inputs=[doc_provider], outputs=[doc_model])

        with gr.Row():
            code_in  = gr.Code(label="Original code (no docstrings)",
                               value=SAMPLE_UNDOCUMENTED, language="python", lines=26)
            code_out = gr.Code(label="Documented code",
                               value="", language="python", lines=26)

        doc_btn = gr.Button("Add Docstrings & Comments")

        doc_btn.click(
            fn=add_docstrings,
            inputs=[doc_provider, doc_model, code_in, doc_language],
            outputs=[code_out]
        )

        # Update the code editor language hint when the user switches language
        doc_language.change(
            fn=lambda lang: gr.Code(language=lang.lower()),
            inputs=[doc_language],
            outputs=[code_in]
        )


ui.launch(inbrowser=True)